# 02 — Scratch CNN Selection and Bayesian Optimization

**Research question:** Which Scratch CNN configurations achieve the strongest **validation Macro F1** within each O/S/T/ST regime?

**Method:** Four independent W&B Bayesian sweeps, each with 30 qualifying trials, fresh model initialization, and validation-only trial and checkpoint selection. Held-out test predictions are generated only after selecting a configuration. This requires your W&B authentication and the real dataset; historical sweep IDs are not reused.

**Important:** Reported legacy scores are not predictions from these new runs, which use the partitions saved by Notebook 01.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import src as mr
ROOT=Path.cwd().resolve()
settings=mr.load_settings(ROOT)
device=mr.setup(settings['seed'])
print('Device:',device,'| W&B project:',settings['wandb_project'])
print('Qualifying trials per regime:',settings['qualifying_trials'])
print('Maximum training epochs:',settings['max_epochs'])


## Architecture, loss and optimization

The scratch network applies configurable blocks of convolution, batch normalization, ReLU and pooling, followed by adaptive global pooling, dropout and a **one-logit** binary classification head. Binary cross-entropy with logits avoids a separate sigmoid in the training loss. Training uses threshold 0.50 for validation decisions, while prediction probabilities are sigmoid-transformed. The notebook caps expanded convolution channels at 256 for practical memory use; this cap is documented rather than concealed.

In the full research profile, each trial can run at most **50 epochs**, with early stopping patience **4** for new trials. Hardware failures are recorded rather than silently treated as successes. The smoke profile reduces epochs, depth and number of trials solely to test the same execution pathway on synthetic files.

**Bayesian search (research profile):** optimizer `{Adam, AdamW, SGD, RMSprop}`; learning rate `1e-5..1e-2` log; batch size `8..256`; zero/positive weight decay with positive value `1e-6..1e-2` log; dropout `0..0.5`; depth `1..7`; and starting filters `8..64`. The goal is **30 qualifying nonzero finite validation Macro F1 trials per regime**, excluding failed attempts. No result is entered in advance.

## Optional audit of one fixed baseline

The configuration below is a deliberately defined **baseline**, not a tuned winner. It can be used to obtain a controlled comparison against the selected model, without peeking at any test result. Model parameter counts are computed directly from the instantiated architecture.

In [ ]:
baseline_config=dict(depth=3,starting_filters=16,dropout=.20,optimizer='adam',learning_rate=1e-3,
                     batch_size=16,weight_decay=0.)
baseline=mr.ScratchCNN(baseline_config['depth'],baseline_config['starting_filters'],baseline_config['dropout'])
print('Baseline trainable parameters:',sum(p.numel() for p in baseline.parameters() if p.requires_grad))
print('Baseline configuration:',baseline_config)
del baseline

## Four independent W&B Bayesian searches

Run this cell after configuring your W&B account. Each regime receives its own independent Bayesian sweep in the configured project, grouped by regime. Selection uses **validation Macro F1**, never held-out test performance. A run qualifies only if it completes with finite, strictly positive validation Macro F1; failed, out-of-memory or zero-F1 runs do not count toward 30. Historical reports record 30 qualifying runs each, but the newly generated results may differ.

**Recorded search space:** optimizer Adam/AdamW/SGD/RMSprop; learning rate log-uniform 1e-5–1e-2; batch size 8–256; dropout 0–0.5; 1–7 convolution blocks; 8–64 starting filters; weight-decay mode zero or positive, with positive values 1e-6–1e-2.


In [ ]:
all_scratch=[]
for regime in mr.REGIMES:
    split=pd.read_csv(Path(settings['out'])/'splits'/f'{regime}_split.csv')
    metrics=mr.tune_one_regime_wandb(split,regime,settings,device)
    all_scratch.append(metrics)
    print(regime,'Bayesian search and held-out evaluation complete')
scratch=pd.DataFrame(all_scratch)
scratch.to_csv(Path(settings['out'])/'scratch_heldout_metrics.csv',index=False)
display(scratch[['regime','val_macro_f1','test_macro_f1','test_accuracy']])


## Plot the evidence rather than paste old sweep results

The following plot reads the trials and training histories just generated by the current run. It highlights the best selected epoch and displays the observed validation objective. In a new experiment the results may differ from another researcher's runs: this is expected and scientifically acceptable.

In [ ]:
fig,axes=plt.subplots(2,4,figsize=(15,6),layout='constrained')
for col,regime in enumerate(mr.REGIMES):
    folder=Path(settings['out'])/'scratch'/regime
    trials=pd.read_csv(folder/'bayesian_trials.csv')
    trials=trials.loc[trials.qualifies == True]
    axes[0,col].scatter(trials['attempt'],trials['validation_macro_f1'],s=18)
    axes[0,col].set(title=f'{regime} Bayesian trials',xlabel='Attempt',ylabel='Validation Macro F1')
    history=pd.read_csv(folder/'selected_training_history.csv')
    axes[1,col].plot(history.epoch,history.train_loss,label='Train BCE')
    axes[1,col].plot(history.epoch,history.validation_loss,label='Validation BCE')
    axes[1,col].set(title=f'{regime} selected training',xlabel='Epoch',ylabel='Loss')
axes[1,0].legend()
out=Path(settings['out'])/'figures'/'bayesian_tuning_and_training.png'
out.parent.mkdir(parents=True,exist_ok=True)
fig.savefig(out,dpi=160);plt.show()
display(scratch[['regime','val_macro_f1','test_macro_f1']])


## Interpretation and handoff

Model comparison must not confuse (a) per-epoch **validation Macro F1** used during Bayesian selection and (b) **held-out Macro F1** used as the final result. Each of the four selected scratch-CNN models is saved under its own regime directory. Notebook 03 will independently fine-tune ResNet18 models using the corresponding splits; it will not continue training an O model on S, T or ST.

### Interpretation

Compare the selected validation objective against the final test Macro F1. Large declines may indicate limited generalization rather than failed training. Report **class-specific flip recall**, especially under T and ST. Interpret exact metrics only after this cell has run; the historic reported held-out Scratch CNN Macro F1 values (O 0.9899, S 0.4400, T 0.8305, ST 0.7906) belong to earlier experiment partitions.
